In [ ]:
import onep
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import scipy.stats
%matplotlib inline

In [ ]:
# #Save all Recall objects as pickle
# import pickle
# # List of objects to save
# astros = [3, 4, 5, 6, 7, 8, 9]  # Add all the astros you want to save to this list
#
# # Loop through each astro and save it
# for i, astro in enumerate(astros, start=1):
#     filename = f'/Users/suthardr/Desktop/astro{i+2}_fc.pkl'
#     with open(filename, 'rb') as filehandler:
#         globals()[f'astro{i+2}fc'] = pickle.load(filehandler)
#     print(f'Astro {i+2} loaded successfully.')

In [ ]:
import pickle

#Load pickles for FC and CxtA
with open('/Users/suthardr/Desktop/collection_fc_allmice.pkl', 'rb') as file:
    collection_fc = pickle.load(file)

with open('/Users/suthardr/Desktop/collection_cxta_allmice.pkl', 'rb') as file:
    collection_cxta = pickle.load(file)
reg_pairs = collection_fc.animals['astroF6'].registration_tables['FC-A']

In [ ]:
table = collection_fc.animals['astroF6'].registration_tables['FC-A']

In [ ]:
print(table)

In [ ]:
#Filter by cells only in both days
both_idxs = table[(table[:, 0] != -1) & (table[:, 1] != -1)]

In [ ]:
both_idxs

In [ ]:
fc_traces = collection_fc.animals['astroF6'].accepted_traces
recall_traces = collection_cxta.animals['astroF6'].accepted_traces

In [ ]:
fc_active = fc_traces[both_idxs[0]]
fc_active

In [ ]:
recall_active = recall_traces[both_idxs[1]]
recall_active

In [ ]:
fc_dfz = stats.zscore(fc_active, axis=1)
fc_dfz

In [ ]:
recall_dfz = stats.zscore(recall_active, axis=1)
recall_dfz

In [ ]:
fc_dfz = fc_dfz[:,:3303]
fc_dfz

In [ ]:
recall_dfz = recall_dfz[:,:3303]
recall_dfz

In [ ]:
across_eta_, time = onep.eta_individual_cells(
    data=fc_dfz,
    timestamps=collection_cxta_fc.animals['astro4'].Timestamps[:3303,].to_numpy().squeeze(),
    events=[[120,180,240,300],], #detected onsets
    window=15
)

In [ ]:
across_eta_R, time_R = onep.eta_individual_cells(
    data=recall_dfz,
    timestamps=collection_cxta_recall.animals['astro4'].Timestamps[:3303,].to_numpy().squeeze(),
    events=[[121.5068386007773, 171.4685321043864, 243.9129876846197,
       285.4811166796225 ],], #detected onsets
    window=15
)

In [ ]:
#Reactivated cells, sorted within recall
# recall_idxs = np.argmax(across_eta_R, axis=1)
# dummy_idxs = np.arange(0, 118, 1) #number of both_idxs
# dummy_table = np.hstack((recall_idxs.reshape(-1, 1), dummy_idxs.reshape(-1, 1)))
# sorted_recall_idxs = np.argsort(dummy_table[:, 0])
#
# sorted_arr = across_eta_R[sorted_recall_idxs]
# # sorted_arr = recall_dfz[sorted_recall_idxs] #plot entire thing
# fig, ax = plt.subplots()
# ax.axvline((time_R.shape[0]/3), linestyle='--', color='white')
# sns.heatmap(sorted_arr, cmap='mako',cbar=True, cbar_kws={"label": r"z-scored $\frac{dF}{F}$"})
# ax.set_xticks(np.linspace(0, sorted_arr.shape[1], 5), np.linspace(np.min(time_R), np.max(time_R), 5))
# ax.set_title('Reactivated: Sorted Recall (Astro3)')
# ax.set_xlabel('Time (seconds)')
# ax.set_ylabel('Cell #')
# fig.savefig('/Users/suthardr/Desktop/Astro4_recall_fc_sorted.png')

In [ ]:
#Reactivated cells, sorted from FC
max_idxs = np.zeros(119)  #change based on # of overlapping cells
for i in range(across_eta_.shape[0]):
    idx = np.argmax(across_eta_[i])
    max_idxs[i] = idx
new_table = np.hstack((both_idxs, max_idxs.reshape(-1, 1)))
fc_idxs = np.argsort(new_table[:, 2])
###################
sorted_arr = across_eta_R[fc_idxs] #plot just the eta times
# sorted_arr = recall_dfz[fc_idxs] #plot entire thing

fig, ax = plt.subplots()
ax.axvline((time_R.shape[0]/3), linestyle='--', color='white')
sns.heatmap(sorted_arr, cmap='mako',cbar=True, cbar_kws={"label": r"z-scored $\frac{dF}{F}$"})
ax.set_xticks(np.linspace(0, sorted_arr.shape[1], 6), np.linspace(np.min(time_R), np.max(time_R), 6))
ax.set_title('Reactivated: Sorted FC (Astro4)') #119 overlapping cells
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Cell #')
fig.savefig('/Users/suthardr/Desktop/Astro4_recall_fc_sorted.png')

In [ ]:
#sorted from FC
# fc_tracesz = stats.zscore(recall_traces, axis=1)
# max_indicesall = np.argmax(fc_tracesz, axis=1)
# # time = np.linspace(-14, 28, 5)
# sorted_ind = np.argsort(max_indicesall)
# sorted_arr = fc_tracesz[sorted_ind]
# fig, ax = plt.subplots()
# # ax.axvline(x=42, color='white', linestyle='--')
# # ax.axvline(x=105, color='white', linestyle='--')
# sns.heatmap(sorted_arr, cmap='mako',cbar=True, cbar_kws={"label": r"z-scored $\frac{dF}{F}$"}, vmin=-5, vmax=10)
# # ax.set_xticks(np.linspace(0, len(fc_tracesz.T), 5), time)
# ax.set_title('Recall: Astro9 Sorted')
# # ax.set_title('Recall Cells: Active FC-Recall Sorted from FC')
# ax.set_xlabel('Time (seconds)')
# ax.set_ylabel('Cell #')